## Preliminaries

Add your own openAI API key in a file named 'openai.key' in this directory before running this notebook

In [1]:
import math
import pickle

import pandas as pd
import numpy as np

from tqdm import tqdm
from openai import OpenAI


In [2]:
data_path = './all_data.csv'
df = pd.read_csv(data_path)
    
label_set = [
    'buddhist', 
    'christian', 
    'hindu', 
    'jewish', 
    'muslim', 
    'other_religion'
]

df

,id,comment_text,split,created_date,publication_id,parent_id,article_id,rating,funny,wow,...,white,asian,latino,other_race_or_ethnicity,physical_disability,intellectual_or_learning_disability,psychiatric_or_mental_illness,other_disability,identity_annotator_count,toxicity_annotator_count
0,1083994,He got his money... now he lies in wait till a...,train,2017-03-06 15:21:53.675241+00,21,NaN,317120,approved,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,67
1,650904,Mad dog will surely put the liberals in mental...,train,2016-12-02 16:44:21.329535+00,21,NaN,154086,approved,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,76
2,5902188,And Trump continues his lifelong cowardice by ...,train,2017-09-05 19:05:32.341360+00,55,NaN,374342,approved,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,63
3,7084460,"""while arresting a man for resisting arrest"".\...",test,2016-11-01 16:53:33.561631+00,13,NaN,149218,approved,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,76
4,5410943,Tucker and Paul are both total bad ass mofo's.,train,2017-06-14 05:08:21.997315+00,21,NaN,344096,approved,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999511,1018736,Another man shamming article. If white men did...,train,2017-02-20 07:20:49.964620+00,54,NaN,169202,approved,0,0,...,0.8,0.0,0.0,0.0,0.000000,0.0,0.0,0.00000,10,10
1999512,340016,"""no matter what is put in front of you regardi...",train,2016-06-06 06:43:04.780968+00,21,339965.0,137961,approved,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.00000,10,10
1999513,919629,The Democrat party aided and abetted by it's M...,train,2017-01-30 02:44:29.168863+00,54,NaN,164845,rejected,0,1,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.00000,11,10
1999514,5165492,I just don't find her a very good representati...,train,2017-04-22 18:42:02.442987+00,54,NaN,328877,approved,1,0,...,0.0,0.0,0.0,0.0,0.003717,0.0,0.0,0.00000,269,10


In [3]:
df.columns

Index(['id', 'comment_text', 'split', 'created_date', 'publication_id',
       'parent_id', 'article_id', 'rating', 'funny', 'wow', 'sad', 'likes',
       'disagree', 'toxicity', 'severe_toxicity', 'obscene', 'sexual_explicit',
       'identity_attack', 'insult', 'threat', 'male', 'female', 'transgender',
       'other_gender', 'heterosexual', 'homosexual_gay_or_lesbian', 'bisexual',
       'other_sexual_orientation', 'christian', 'jewish', 'muslim', 'hindu',
       'buddhist', 'atheist', 'other_religion', 'black', 'white', 'asian',
       'latino', 'other_race_or_ethnicity', 'physical_disability',
       'intellectual_or_learning_disability', 'psychiatric_or_mental_illness',
       'other_disability', 'identity_annotator_count',
       'toxicity_annotator_count'],
      dtype='object')

In [4]:
print(df["toxicity"])

0          0.373134
1          0.605263
2          0.666667
3          0.815789
4          0.550000
             ...   
1999511    0.400000
1999512    0.400000
1999513    0.400000
1999514    0.400000
1999515    0.400000
Name: toxicity, Length: 1999516, dtype: float64


In [5]:
df = df[['comment_text', 'split', 'toxicity'] + label_set]
df

,comment_text,split,toxicity,buddhist,christian,hindu,jewish,muslim,other_religion
0,He got his money... now he lies in wait till a...,train,0.373134,NaN,NaN,NaN,NaN,NaN,NaN
1,Mad dog will surely put the liberals in mental...,train,0.605263,NaN,NaN,NaN,NaN,NaN,NaN
2,And Trump continues his lifelong cowardice by ...,train,0.666667,NaN,NaN,NaN,NaN,NaN,NaN
3,"""while arresting a man for resisting arrest"".\...",test,0.815789,NaN,NaN,NaN,NaN,NaN,NaN
4,Tucker and Paul are both total bad ass mofo's.,train,0.550000,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
1999511,Another man shamming article. If white men did...,train,0.400000,0.00000,0.00000,0.0,0.0,0.000000,0.0
1999512,"""no matter what is put in front of you regardi...",train,0.400000,0.00000,0.00000,0.0,0.0,0.000000,0.0
1999513,The Democrat party aided and abetted by it's M...,train,0.400000,0.00000,0.00000,0.0,0.0,0.000000,0.0
1999514,I just don't find her a very good representati...,train,0.400000,0.00000,0.00000,0.0,0.0,0.000000,0.0


In [6]:
df = df[
    ((df['buddhist'].notna()) & (df['buddhist'] > 0)) |
    ((df['christian'].notna()) & (df['christian'] > 0)) |
    ((df['hindu'].notna()) & (df['hindu'] > 0)) |
    ((df['jewish'].notna()) & (df['jewish'] > 0)) |
    ((df['muslim'].notna()) & (df['muslim'] > 0)) |
    ((df['other_religion'].notna()) & (df['other_religion'] > 0)) 
    ]

df.fillna(0, inplace=True)

df

/tmp/ipykernel_740349/3609150174.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.fillna(0, inplace=True)


,comment_text,split,toxicity,buddhist,christian,hindu,jewish,muslim,other_religion
7678,OH yes - Were those evil Christian Missionarie...,train,0.800000,0.00000,1.00000,0.0,0.0,0.000000,0.0
7701,The robot censor seems disinclined to accept s...,train,0.800000,0.00000,0.00000,0.0,0.0,1.000000,0.0
7704,Agreed: there's no equivalence. What is stoppi...,train,0.800000,0.00000,1.00000,0.0,0.0,0.000000,0.0
7716,It took them long enough. And it goes against ...,train,0.181818,0.00000,1.00000,0.0,0.0,0.000000,0.0
7719,C.Parsons\nIf the terrorists are loathsome for...,train,0.671429,0.00000,0.00000,0.0,0.0,1.000000,0.0
...,...,...,...,...,...,...,...,...,...
1999498,First we needed strength. Then diversity. Th...,train,0.400000,0.00000,0.00000,0.0,0.0,0.600000,0.1
1999502,"When you say speaking in code, you are adding ...",train,0.400000,0.10000,0.10000,0.0,0.0,0.800000,0.0
1999503,I don't think I can picture Christ ever saying...,train,0.400000,0.00000,0.50000,0.0,0.0,0.000000,0.0
1999506,"What about the Mormons and the ""Polynesian Cul...",train,0.400000,0.00000,0.20000,0.0,0.0,0.000000,0.3


In [7]:
df_train = df[df['split'] == 'train']
df_train

,comment_text,split,toxicity,buddhist,christian,hindu,jewish,muslim,other_religion
7678,OH yes - Were those evil Christian Missionarie...,train,0.800000,0.00000,1.00000,0.0,0.0,0.000000,0.0
7701,The robot censor seems disinclined to accept s...,train,0.800000,0.00000,0.00000,0.0,0.0,1.000000,0.0
7704,Agreed: there's no equivalence. What is stoppi...,train,0.800000,0.00000,1.00000,0.0,0.0,0.000000,0.0
7716,It took them long enough. And it goes against ...,train,0.181818,0.00000,1.00000,0.0,0.0,0.000000,0.0
7719,C.Parsons\nIf the terrorists are loathsome for...,train,0.671429,0.00000,0.00000,0.0,0.0,1.000000,0.0
...,...,...,...,...,...,...,...,...,...
1999498,First we needed strength. Then diversity. Th...,train,0.400000,0.00000,0.00000,0.0,0.0,0.600000,0.1
1999502,"When you say speaking in code, you are adding ...",train,0.400000,0.10000,0.10000,0.0,0.0,0.800000,0.0
1999503,I don't think I can picture Christ ever saying...,train,0.400000,0.00000,0.50000,0.0,0.0,0.000000,0.0
1999506,"What about the Mormons and the ""Polynesian Cul...",train,0.400000,0.00000,0.20000,0.0,0.0,0.000000,0.3


In [8]:
df_test = df[df['split'] == 'test']
df_test

,comment_text,split,toxicity,buddhist,christian,hindu,jewish,muslim,other_religion
7796,"If the ""precedent"" [sic] is a ""perfectionist"",...",test,0.515152,0.0,1.0,0.0,0.0,0.0,0.0
8415,White evangelical Christians worship the Pussy...,test,0.750000,0.0,1.0,0.0,0.0,0.0,0.0
8539,... super fakey bigot racist sexist 'christian...,test,0.662500,0.0,1.0,0.0,0.0,0.0,0.0
8561,and the worthless criminal illegals that you l...,test,0.857143,0.0,0.0,0.0,0.0,1.0,0.0
8640,"Saunders offers the phrases : ""(terms that, i...",test,0.900000,0.0,0.0,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...
1999418,Right on Dennis. Trump maybe doing nothing mo...,test,0.400000,0.0,0.1,0.0,0.0,0.0,0.0
1999422,"I share the common outrage about this ""new pol...",test,0.400000,0.0,0.3,0.0,0.0,0.0,0.0
1999453,"Herod's ""slaughter of the innocents"" in Matthe...",test,0.400000,0.0,0.6,0.0,0.0,0.0,0.0
1999492,My son shouldn't have to be afraid during THOS...,test,0.400000,0.0,0.0,0.0,0.0,1.0,0.0


In [9]:
def read_key(filename):
    with open(filename) as f:
        return f.read()

client = OpenAI(api_key=read_key('openai.key'))

In [10]:
def embed(text, model="text-embedding-3-large"):
    text = text.replace("\n", " ")
    return client.embeddings.create(input = [text], model=model, dimensions=512).data[0].embedding

In [ ]:
df_train["embedded"] = df_train["comment_text"].apply(embed)
df_train.reset_index(drop=True)
# save the result
filename = 'jigsaw_train.pkl'
with open(filename, "wb") as file:
    pickle.dump(df_train, file)

df_train

In [ ]:
df_test["embedded"] = df_test["comment_text"].apply(embed)
df_test.reset_index(drop=True)
# save the result
filename = 'jigsaw_test.pkl'
with open(filename, "wb") as file:
    pickle.dump(df_test, file)

df_test


In [32]:
with open("jigsaw_train.pkl", "rb") as file:
    df_train = pickle.load(file)
with open("jigsaw_test.pkl", "rb") as file:
    df_test = pickle.load(file)

df_train

,comment_text,split,buddhist,christian,hindu,jewish,muslim,other_religion,embedded
7678,OH yes - Were those evil Christian Missionarie...,train,0.00000,1.00000,0.0,0.0,0.000000,0.0,"[0.015273211523890495, -0.0038294349797070026,..."
7701,The robot censor seems disinclined to accept s...,train,0.00000,0.00000,0.0,0.0,1.000000,0.0,"[-0.013876539655029774, -0.11917202919721603, ..."
7704,Agreed: there's no equivalence. What is stoppi...,train,0.00000,1.00000,0.0,0.0,0.000000,0.0,"[0.053906071931123734, 0.03860460966825485, -0..."
7716,It took them long enough. And it goes against ...,train,0.00000,1.00000,0.0,0.0,0.000000,0.0,"[0.011510981246829033, -0.0005849719163961709,..."
7719,C.Parsons\nIf the terrorists are loathsome for...,train,0.00000,0.00000,0.0,0.0,1.000000,0.0,"[-0.013909636065363884, -0.050814393907785416,..."
...,...,...,...,...,...,...,...,...,...
1999498,First we needed strength. Then diversity. Th...,train,0.00000,0.00000,0.0,0.0,0.600000,0.1,"[0.023074055090546608, -0.06494539231061935, -..."
1999502,"When you say speaking in code, you are adding ...",train,0.10000,0.10000,0.0,0.0,0.800000,0.0,"[0.005696144420653582, 0.01229139231145382, -0..."
1999503,I don't think I can picture Christ ever saying...,train,0.00000,0.50000,0.0,0.0,0.000000,0.0,"[-0.030144574120640755, 0.008785936050117016, ..."
1999506,"What about the Mormons and the ""Polynesian Cul...",train,0.00000,0.20000,0.0,0.0,0.000000,0.3,"[0.003190038725733757, -0.06372644007205963, -..."


In [ ]:
# the toxicity column has been removed, we need to add it back
df_train["toxicity"] = df[df['split'] == 'train']["toxicity"].values
df_test["toxicity"] = df[df['split'] == 'test']["toxicity"].values

In [ ]:
# dump the result
filename = 'jigsaw_train.pkl'
with open(filename, "wb") as file:
    pickle.dump(df_train, file)
filename = 'jigsaw_test.pkl'
with open(filename, "wb") as file:
    pickle.dump(df_test, file)

In [43]:
print(df_train.columns)
print(df_train)

Index(['comment_text', 'split', 'buddhist', 'christian', 'hindu', 'jewish',
       'muslim', 'other_religion', 'embedded', 'toxicity'],
      dtype='object')
                                              comment_text  split  buddhist  \
7678     OH yes - Were those evil Christian Missionarie...  train   0.00000   
7701     The robot censor seems disinclined to accept s...  train   0.00000   
7704     Agreed: there's no equivalence. What is stoppi...  train   0.00000   
7716     It took them long enough. And it goes against ...  train   0.00000   
7719     C.Parsons\nIf the terrorists are loathsome for...  train   0.00000   
...                                                    ...    ...       ...   
1999498  First we needed strength.  Then diversity.  Th...  train   0.00000   
1999502  When you say speaking in code, you are adding ...  train   0.10000   
1999503  I don't think I can picture Christ ever saying...  train   0.00000   
1999506  What about the Mormons and the "Polynesian 

In [57]:
# Vectorized approach: significantly faster and cleaner
religion_labels = df_train[label_set].idxmax(axis=1).tolist()

print(religion_labels[:10])

['christian', 'muslim', 'christian', 'christian', 'muslim', 'muslim', 'muslim', 'christian', 'muslim', 'muslim']


In [59]:
from collections import Counter
count_labels = Counter(religion_labels)
print(count_labels)

Counter({'christian': 57968, 'muslim': 21452, 'jewish': 6531, 'other_religion': 3506, 'hindu': 743, 'buddhist': 740})


In [60]:
# selct only the rows with religion labels ["christian", "muslim", "jewish"]
df_train_religion = df_train[df_train[label_set].idxmax(axis=1).isin(["christian", "muslim", "jewish", "hindu", "buddhist"])]
religion_labels = df_train_religion[label_set].idxmax(axis=1).tolist()
print("Number of training samples:", len(religion_labels))
print(Counter(religion_labels))

Number of training samples: 87434
Counter({'christian': 57968, 'muslim': 21452, 'jewish': 6531, 'hindu': 743, 'buddhist': 740})


In [63]:
df_test_religion = df_test[df_test[label_set].idxmax(axis=1).isin(["christian", "muslim", "jewish", "hindu", "buddhist"])]
test_religion_labels = df_test_religion[label_set].idxmax(axis=1).tolist()
print("Number of test samples:", len(test_religion_labels))
print(Counter(test_religion_labels))

Number of test samples: 9058
Counter({'christian': 6092, 'muslim': 2089, 'jewish': 711, 'hindu': 90, 'buddhist': 76})


In [49]:
# transform the embedded column into a numpy array
x = np.array(df_train_religion["embedded"].tolist())
x_test = np.array(df_test_religion["embedded"].tolist())
print(x.shape, x_test.shape)

(85951, 512) (8892, 512)


In [50]:
z_values, z = np.unique(religion_labels, return_inverse=True)
z_test = np.searchsorted(z_values, test_religion_labels)
print(z_values)
print(z[:10])
print(z_test[:10])

['christian' 'jewish' 'muslim']
[0 2 0 0 2 2 2 0 2 2]
[0 0 0 2 1 2 2 1 0 1]


In [53]:
# get the toxicity labels (with a 0.5 threshold)
# assign the value "toxic" to 1 and "non-toxic" to 0
# y = (df_train_religion["toxicity"] >= 0.5).astype(int).values
# y_test = (df_test_religion["toxicity"] >= 0.5).astype(int).values
y = (df_train_religion["toxicity"] >= 0.5).astype(int).values
y_test = (df_test_religion["toxicity"] >= 0.5).astype(int).values
y = ["toxic" if label == 1 else "non-toxic" for label in y]
y_test = ["toxic" if label == 1 else "non-toxic" for label in y_test]
print(y[:10])
print(y_test[:10])

['toxic', 'toxic', 'toxic', 'non-toxic', 'toxic', 'non-toxic', 'non-toxic', 'toxic', 'toxic', 'toxic']
['toxic', 'toxic', 'toxic', 'toxic', 'toxic', 'toxic', 'toxic', 'non-toxic', 'toxic', 'toxic']


In [39]:
from sklearn.neural_network import MLPClassifier
clf = MLPClassifier(random_state=0, max_iter=300, early_stopping=True, validation_fraction=0.1).fit(x, z)
print(clf.score(x, z))
print(clf.score(x_test, z_test))

0.9565333736663914
0.9433198380566802


In [55]:
y = [1 if label == "toxic" else 0 for label in y]
y_test = [1 if label == "toxic" else 0 for label in y_test]
clf_y = MLPClassifier(random_state=0, max_iter=300, early_stopping=True, validation_fraction=0.1).fit(x, y)
print(clf_y.score(x, y))
print(clf_y.score(x_test, y_test))

0.9120196390966946
0.898110661268556
